In [1]:
import os
import json
from dotenv import load_dotenv
from typing_extensions import TypedDict, Annotated
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
from tools.redmine import get_issues, get_members, get_versions, get_projects

**AgentState:** State tracks the **plan**, current **step**, and **messages**

In [2]:
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    plan: list[str]
    index: int
    observations: list[str]

In [3]:
load_dotenv()

True

**Model Declaration:**

In [4]:
llm = ChatOpenAI(
    model="openrouter/auto",
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0
)

**Tool Calling:**

In [5]:
tools = [get_projects, get_issues, get_members, get_versions]
TOOL_REGISTRY = {tool.name: tool for tool in tools}

**Planner**, generates the plan

In [6]:
PLANNER_PROMPT = """You are a planning assistant for a Redmine project management chatbot.
You have access to ONLY these tools:
- get_projects : retrieves all available projects
- get_issues   : retrieves issues with filters (project_id, status_id, priority_id, due_before, assigned_to_id)
- get_members  : retrieves members of a project
- get_versions : retrieves sprints/versions of a project
 
Break the user request into an ordered list of tool calls needed to answer it.
Each step must specify exactly which tool to call and with which parameters.
 
Respond ONLY with a valid JSON array of objects, like this:
[
  {"tool": "get_projects", "args": {}},
  {"tool": "get_issues", "args": {"project_id": "e-commerce-platform", "status_id": "open"}},
  {"tool": "get_members", "args": {"project_id": "e-commerce-platform"}}
]
 
Rules:
- Only use tool names from the list above
- Always include "tool" and "args" keys in each step
- If you don't know the project_id yet, add get_projects as the first step
- Never add steps like "parse response" or "summarize" — only tool calls
"""

In [7]:
def planner(state: AgentState) -> AgentState:
    user_message = state["messages"][-1]
    
    response = llm.invoke([
        ("system", PLANNER_PROMPT),
        ("human", f"User request: {user_message.content}")
    ])
    
    # Parse the JSON plan robustly
    try:
        raw = response.content.strip()
        # Strip markdown code blocks if present
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        plan = json.loads(raw.strip())
    except json.JSONDecodeError:
        # Fallback — if parsing fails, default to getting projects
        print(f"[Planner] Failed to parse plan, using fallback. Raw: {response.content}")
        plan = [{"tool": "get_projects", "args": {}}]
    
    print(f"\n PLAN ({len(plan)} steps):")
    for i, step in enumerate(plan):
        print(f"   Step {i+1}: {step['tool']}({step.get('args', {})})")
    print()
    
    return {
        "plan": plan,
        "index": 0,
        "observations": []
    }

**Executor** processes one step, calling tools if needed

In [8]:
def executor(state: AgentState) -> AgentState:
    plan = state["plan"]
    index = state["index"]
    
    if index >= len(plan):
        return {"index": index}
    
    step = plan[index]
    tool_name = step.get("tool", "")
    tool_args = step.get("args", {})
    
    print(f"STEP {index + 1}/{len(plan)} — Executing: {tool_name}({tool_args})")
    
    if tool_name in TOOL_REGISTRY:
        try:
            tool_func = TOOL_REGISTRY[tool_name]
            result = tool_func.invoke(tool_args)
            print(f"Result: {json.dumps(result, ensure_ascii=False)[:200]}...")
        except Exception as e:
            result = {"error": str(e)}
            print(f"Tool error: {e}")
    else:
        result = {"error": f"Unknown tool: {tool_name}"}
        print(f"Unknown tool: {tool_name}")
    
    # Accumulate observations — preserve all previous ones
    previous_observations = state.get("observations", [])
    new_observation = {
        "step": index + 1,
        "tool": tool_name,
        "args": tool_args,
        "result": result
    }
    
    return {
        "observations": previous_observations + [new_observation],
        "index": index + 1
    }

In [9]:
def should_continue(state: AgentState) -> str:
    if state["index"] < len(state["plan"]):
        return "executor"
    return "aggregator"

**Aggregation Agent** collects the executor's output into one cohesive response

In [10]:
AGGREGATOR_PROMPT = """You are a project management assistant.
You have executed a series of Redmine API calls to answer the user's question.
Below are all the results collected from those calls.
Write a clear, structured, and complete answer in French based strictly on this data.
Do not invent any data — only use what is provided in the observations."""

In [11]:
def aggregator(state: AgentState) -> dict:
    observations = state.get("observations", [])
    user_question = state["messages"][0].content
    
    print(f"\nAGGREGATING {len(observations)} observations...")
    
    # Format all observations for the LLM
    formatted_observations = ""
    for obs in observations:
        formatted_observations += f"\nStep {obs['step']} — {obs['tool']}({obs['args']}):\n"
        formatted_observations += json.dumps(obs["result"], ensure_ascii=False, indent=2)
        formatted_observations += "\n"
    
    response = llm.invoke([
        ("system", AGGREGATOR_PROMPT),
        ("human", f"""User question: {user_question}
 
Data collected from Redmine:
{formatted_observations}
 
Now write the final answer for the user.""")
    ])
    
    print("\n─" * 25)
    print("FINAL ANSWER:")
    print(response.content)
    print("─" * 25)
    
    return {
        "messages": [response],
        "observations": observations
    }

**Build the Graph**

In [12]:
builder = StateGraph(AgentState)
 
builder.add_node("planner", planner)
builder.add_node("executor", executor)
builder.add_node("aggregator", aggregator)
 
builder.set_entry_point("planner")
builder.add_edge("planner", "executor")
builder.add_conditional_edges(
    "executor",
    should_continue,
    {
        "executor": "executor",
        "aggregator": "aggregator"
    }
)
builder.add_edge("aggregator", END)
 
app = builder.compile()

**Example 1**

In [13]:
result = app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Give me a list of all open issues in the e-commerce-platform project and the members assigned to them."
            }
        ],
        "observations": []
    }
)


 PLAN (2 steps):
   Step 1: get_issues({'project_id': 'e-commerce-platform', 'status_id': 'open'})
   Step 2: get_members({'project_id': 'e-commerce-platform'})

STEP 1/2 — Executing: get_issues({'project_id': 'e-commerce-platform', 'status_id': 'open'})
Result: {"total_count": 18, "issues": [{"id": 32, "subject": "End-to-end testing", "status": "New", "priority": "Normal", "assigned_to": "Lina Saidi", "due_date": "2026-06-01", "project": "E-Commerce Platform...
STEP 2/2 — Executing: get_members({'project_id': 'e-commerce-platform'})
Result: {"total_count": 5, "members": [{"id": 9, "name": "Nour Belhaj", "roles": ["Developer", "Manager"]}, {"id": 10, "name": "Omar Cherif", "roles": ["Developer"]}, {"id": 11, "name": "Lina Saidi", "roles":...

AGGREGATING 2 observations...

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
FINAL ANSWER:
Voici la liste des tickets ouverts pour le projet "E-Commerce Platform" et les membres qui leur sont assignés :

*   **End-to-end testing** (ID: 32) - 

In [14]:
result

{'messages': [HumanMessage(content='Give me a list of all open issues in the e-commerce-platform project and the members assigned to them.', additional_kwargs={}, response_metadata={}, id='73255b02-c642-4382-a230-b0d19c368fbf'),
  AIMessage(content='Voici la liste des tickets ouverts pour le projet "E-Commerce Platform" et les membres qui leur sont assignés :\n\n*   **End-to-end testing** (ID: 32) - Assigné à : Lina Saidi\n*   **Production deployment** (ID: 31) - Assigné à : Tarek Mejri\n*   **CI/CD pipeline setup** (ID: 30) - Assigné à : Omar Cherif\n*   **Admin dashboard** (ID: 29) - Assigné à : Tarek Mejri\n*   **React.js storefront UI** (ID: 28) - Assigné à : Lina Saidi\n*   **Payment webhook handling** (ID: 27) - Assigné à : Omar Cherif\n*   **Email notifications (order confirm)** (ID: 26) - Assigné à : Tarek Mejri\n*   **Order management API** (ID: 25) - Assigné à : Lina Saidi\n*   **Stripe payment integration** (ID: 24) - Assigné à : Omar Cherif\n*   **Product image upload (S3)*

In [15]:
print(result["messages"][-1].content)

Voici la liste des tickets ouverts pour le projet "E-Commerce Platform" et les membres qui leur sont assignés :

*   **End-to-end testing** (ID: 32) - Assigné à : Lina Saidi
*   **Production deployment** (ID: 31) - Assigné à : Tarek Mejri
*   **CI/CD pipeline setup** (ID: 30) - Assigné à : Omar Cherif
*   **Admin dashboard** (ID: 29) - Assigné à : Tarek Mejri
*   **React.js storefront UI** (ID: 28) - Assigné à : Lina Saidi
*   **Payment webhook handling** (ID: 27) - Assigné à : Omar Cherif
*   **Email notifications (order confirm)** (ID: 26) - Assigné à : Tarek Mejri
*   **Order management API** (ID: 25) - Assigné à : Lina Saidi
*   **Stripe payment integration** (ID: 24) - Assigné à : Omar Cherif
*   **Product image upload (S3)** (ID: 23) - Assigné à : Tarek Mejri
*   **Cart persistence with Redis** (ID: 22) - Assigné à : Omar Cherif
*   **Shopping cart API** (ID: 21) - Assigné à : Lina Saidi
*   **Product search & filters** (ID: 20) - Assigné à : Tarek Mejri
*   **Product catalog API

**Example 2**

In [21]:
result = app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What projects are currently active?"
            }
        ],
        "observations": []
    }
)


 PLAN (1 steps):
   Step 1: get_projects({})

STEP 1/1 — Executing: get_projects({})
Result: {"total_count": 2, "projects": [{"id": 1, "name": "AI Chatbot Platform", "identifier": "ai-chatbot-platform", "description": "Development of an AI-powered project management assistant with Redmine int...

AGGREGATING 1 observations...

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
FINAL ANSWER:
Voici les projets actuellement actifs :

*   **AI Chatbot Platform** (Identifiant : ai-chatbot-platform)
    *   Description : Development of an AI-powered project management assistant with Redmine integration
*   **E-Commerce Platform** (Identifiant : e-commerce-platform)
    *   Description : Un projet de développement d'une plateforme e-commerce avec backend, frontend, paiement et déploiement. C'est un bon cas pour Plan & Execute car les questions de type "donne moi un rapport complet" ou "analyse les risques du projet" nécessitent plusieurs appels Redmine enchaînés.
─────────────────────────


In [22]:
print(result["messages"][-1].content)

Voici les projets actuellement actifs :

*   **AI Chatbot Platform** (Identifiant : ai-chatbot-platform)
    *   Description : Development of an AI-powered project management assistant with Redmine integration
*   **E-Commerce Platform** (Identifiant : e-commerce-platform)
    *   Description : Un projet de développement d'une plateforme e-commerce avec backend, frontend, paiement et déploiement. C'est un bon cas pour Plan & Execute car les questions de type "donne moi un rapport complet" ou "analyse les risques du projet" nécessitent plusieurs appels Redmine enchaînés.


**Exemple 3**

In [23]:
result = app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What tasks are due next week and who is assigned to them?"
            }
        ],
        "observations": []
    }
)
print(result["messages"][-1].content)


 PLAN (2 steps):
   Step 1: get_projects({})
   Step 2: get_issues({'due_before': 'next week'})

STEP 1/2 — Executing: get_projects({})
Result: {"total_count": 2, "projects": [{"id": 1, "name": "AI Chatbot Platform", "identifier": "ai-chatbot-platform", "description": "Development of an AI-powered project management assistant with Redmine int...
STEP 2/2 — Executing: get_issues({'due_before': 'next week'})
Tool error: 1 validation error for get_issues
project_id
  Field required [type=missing, input_value={'due_before': 'next week'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

AGGREGATING 2 observations...

─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
─
FINAL ANSWER:
Je suis désolé, mais je n'ai pas pu récupérer les informations sur les tâches à venir la semaine prochaine. L'API Redmine nécessite que je spécifie un projet pour rechercher les tâches. Pourriez-vous me dire dans quel projet vous souhaitez que je recherche les tâches ?

**Mermaid graph visualisation:**

In [16]:
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	executor(executor)
	aggregator(aggregator)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	executor -.-> aggregator;
	planner --> executor;
	aggregator --> __end__;
	executor -.-> executor;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [20]:
from IPython.display import Image, display
import base64

def mm(graph):
    graphbytes = graph.encode("utf8")
    base64_bytes = base64.urlsafe_b64encode(graphbytes)
    base64_string = base64_bytes.decode("ascii")
    display(Image(url='https://mermaid.ink/img/' + base64_string))

mermaid_graph = """
graph TD;
__start__([<p>__start__</p>]):::first
planner(planner)
executor(executor)
aggregator(aggregator)
__end__([<p>__end__</p>]):::last
__start__ --> planner;
executor -.-> aggregator;
planner --> executor;
aggregator --> __end__;
executor -.-> executor;
classDef default fill:#f2f0ff,line-height:1.2
classDef first fill-opacity:0
classDef last fill:#bfb6fc
"""

# Example Usage
mm(mermaid_graph)
